In [1]:
import os
import random
import numpy as np
import warnings
import tensorflow as tf

warnings.filterwarnings("ignore", category=UserWarning)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# --- KHÓA HẠT GIỐNG TOÀN CỤC ---
SEED = 123
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

print(f"Đã khóa hạt giống toàn cục: {SEED}")

2026-03-23 14:19:53.140865: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774275593.725181      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774275593.838974      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774275594.992983      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774275594.993031      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774275594.993034      24 computation_placer.cc:177] computation placer alr

Đã khóa hạt giống toàn cục: 123


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import os

# 1. Định nghĩa tham số cố định
TRAIN_PATH = '/kaggle/input/datasets/jaquy0802/edge-ai/kaggle_testing/train'
IMG_SIZE = (32, 32)
BATCH_SIZE = 32
NUM_CLASSES = 10

print("Đã sẵn sàng thiết lập!")

Đã sẵn sàng thiết lập!


In [3]:
# Nạp tập Train (80% để huấn luyện)
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb'
)

# Nạp tập Validation (20% để kiểm tra độ chính xác)
val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb'
)

class_names = train_ds.class_names
print(f"Danh sách nhãn: {class_names}")

# Tối ưu tốc độ nạp dữ liệu vào RAM
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

Found 2000 files belonging to 10 classes.
Using 1600 files for training.


I0000 00:00:1774275640.187911      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774275640.194463      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 2000 files belonging to 10 classes.
Using 400 files for validation.
Danh sách nhãn: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']


In [4]:
from tensorflow.keras import layers, models, callbacks

# 1. Đã bỏ RandomFlip. Chỉ giữ xoay và zoom nhẹ.
# Truyền thêm tham số seed để góc xoay cũng cố định sau mỗi lần Run All
data_augmentation = tf.keras.Sequential([
  layers.RandomRotation(0.15, seed=SEED),
  layers.RandomZoom(0.15, seed=SEED),
])

# 2. Cấu trúc mô hình bền vững (~189,450 tham số)
model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    layers.Rescaling(1./255),
    
    # Block 1
    layers.Conv2D(32, (3, 3), padding='same'),
    layers.BatchNormalization(), 
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Block 2
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Block 3
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'), 
    layers.Dropout(0.3), 
    layers.Dense(10, activation='softmax')
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 8, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 189,450 (740.04 KB)

 Trainable params: 189,130 (738.79 KB)

 Non-trainable params: 320 (1.25 KB)

In [5]:
# Tự động giảm tốc độ học khi độ chính xác chững lại
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.00001)

# Dừng sớm nếu không còn tiến bộ và lấy lại bản tốt nhất
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# Mạnh dạn nâng Epochs lên 50, Early Stopping sẽ lo việc dừng đúng lúc
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[reduce_lr, early_stop]
)

Epoch 1/50


I0000 00:00:1774275651.250282      79 cuda_dnn.cc:529] Loaded cuDNN version 91002


50/50 ━━━━━━━━━━━━━━━━━━━━ 12s 35ms/step - accuracy: 0.2640 - loss: 2.1970 - val_accuracy: 0.1400 - val_loss: 2.2484 - learning_rate: 0.0010
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.5023 - loss: 1.3508 - val_accuracy: 0.2300 - val_loss: 2.1403 - learning_rate: 0.0010
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6680 - loss: 0.9037 - val_accuracy: 0.2275 - val_loss: 2.0067 - learning_rate: 0.0010
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8247 - loss: 0.5629 - val_accuracy: 0.3250 - val_loss: 1.7840 - learning_rate: 0.0010
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8425 - loss: 0.4942 - val_accuracy: 0.5950 - val_loss: 1.3225 - learning_rate: 0.0010
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9080 - loss: 0.2850 - val_accuracy: 0.7100 - val_loss: 0.8817 - learning_rate: 0.0010
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9388 - loss: 0.2250 - val_accuracy: 0.8

In [6]:
# 1. Lưu file gốc .h5
model.save('edge_ai_model_final.h5')

# 2. Tạo hàm trích xuất dữ liệu đại diện cho bộ Quantization
def representative_data_gen():
    # Lấy 100 batch đầu tiên từ tập train để tính toán biên độ số nguyên
    for input_value, _ in train_ds.take(100):
        yield [input_value]

# 3. Chuyển đổi sang .tflite chuẩn INT8 cho Edge Device
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
# Bắt buộc phải có dòng này để chuyển hẳn sang INT8
converter.representative_dataset = representative_data_gen 

tflite_model = converter.convert()

# Lưu file .tflite
with open('model_quantized.tflite', 'wb') as f:
    f.write(tflite_model)

print("Đã lưu thành công: edge_ai_model_final.h5 và model_quantized.tflite (Chuẩn INT8)!")

INFO:tensorflow:Assets written to: /tmp/tmpeldobi5a/assets


INFO:tensorflow:Assets written to: /tmp/tmpeldobi5a/assets


Saved artifact at '/tmp/tmpeldobi5a'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  137873144987472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137873144976336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137873138895568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137873138895760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137873138892880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137873138893456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137873138894608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137873138896720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137873138897104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137873138898064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137873138897872

W0000 00:00:1774275681.569825      24 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1774275681.569867      24 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1774275681.585385      24 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled


Đã lưu thành công: edge_ai_model_final.h5 và model_quantized.tflite (Chuẩn INT8)!


fully_quantize: 0, inference_type: 6, input_inference_type: FLOAT32, output_inference_type: FLOAT32


In [7]:
import numpy as np
import os
import cv2
import pandas as pd
import tensorflow as tf

# 1. Định nghĩa đường dẫn và cấu hình (Sửa lại cho đúng Dataset của bạn)
TEST_DIR = '/kaggle/input/datasets/jaquy0802/edge-ai/kaggle_testing/test'
IMG_SIZE = (32, 32) # Phải trùng với kích thước lúc train

print("Đã thiết lập đường dẫn tập Test!")

Đã thiết lập đường dẫn tập Test!


In [8]:
import cv2
import numpy as np
import os

# 1. Sắp xếp file theo số (0.png, 1.png...) để khớp với ID nộp bài
test_files = sorted(os.listdir(TEST_DIR), key=lambda x: int(x.split('.')[0]))

X_test = []

print("Bắt đầu xử lý lại ảnh Test (Đã bỏ chia 255)...")

for f in test_files:
    img_path = os.path.join(TEST_DIR, f)
    img = cv2.imread(img_path)
    if img is not None:
        # Chuyển từ BGR sang RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # Resize về đúng kích thước 32x32
        img = cv2.resize(img, IMG_SIZE)
        
        # QUAN TRỌNG: Giữ nguyên giá trị pixel 0-255. 
        # Không chia cho 255 ở đây vì mô hình đã có lớp Rescaling(1./255) rồi.
        X_test.append(img)

# Chuyển thành mảng Numpy
X_test = np.array(X_test)

print(f"Thành công: Đã xử lý {len(X_test)} ảnh Test.")
print(f"Kích thước mảng đầu vào: {X_test.shape}") # Kết quả nên là (1000, 32, 32, 3)

Bắt đầu xử lý lại ảnh Test (Đã bỏ chia 255)...
Thành công: Đã xử lý 1000 ảnh Test.
Kích thước mảng đầu vào: (1000, 32, 32, 3)


In [9]:
import tensorflow as tf
import numpy as np
import pandas as pd
import os

print("Đang dự đoán bằng .tflite và khớp ID theo tên file thực tế...")

# 1. Nạp mô hình TFLite
interpreter = tf.lite.Interpreter(model_path="model_quantized.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# 2. Chuẩn bị danh sách ID thực tế từ tên file đã sắp xếp
# Giả sử test_files đã được sorted ở ô trước đó
actual_ids = []
for f in test_files:
    file_name = f.split('.')[0] # Lấy phần số: "7" từ "7.png"
    actual_ids.append(f"{int(file_name):05d}") # Chuyển thành "00007"

final_labels_tflite = []

# 3. Chạy dự đoán từng ảnh
for i in range(len(X_test)):
    input_data = np.expand_dims(X_test[i], axis=0).astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    
    output_data = interpreter.get_tensor(output_details[0]['index'])
    label = np.argmax(output_data)
    final_labels_tflite.append(label)

# 4. Tạo file submission với ID chuẩn từ tên file
submission = pd.DataFrame({
    'Id': actual_ids, # Dùng danh sách ID thực tế thay vì dùng range()
    'Label': final_labels_tflite
})

submission.to_csv('submission.csv', index=False)

print("--- THÀNH CÔNG ---")
print(f"Đã tạo xong file submission.csv với {len(submission)} dòng.")
print("Ví dụ 5 dòng đầu tiên:")
print(submission.head())

Đang dự đoán bằng .tflite và khớp ID theo tên file thực tế...


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


--- THÀNH CÔNG ---
Đã tạo xong file submission.csv với 1000 dòng.
Ví dụ 5 dòng đầu tiên:
      Id  Label
0  00000      4
1  00007      2
2  00012      2
3  00014      5
4  00022      1
